In [5]:
import os
import dotenv
from flashboot_core.utils import project_utils
from loguru import logger

root_path = project_utils.get_root_path()
dotenv.load_dotenv(os.path.join(root_path, ".env"), override=True)

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]

In [6]:
import bs4
import chromadb
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.load import dumps, loads

In [7]:
loader = WebBaseLoader(
   web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-title", "post-header","post-content")
        )
    ),
)
blog_docs = loader.load()
doc = blog_docs[0]

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = splitter.split_documents(blog_docs)

valid_docs = [d for d in splits if d.page_content and d.page_content.strip()]
logger.info(f"Total splits: {len(splits)}, Valid splits: {len(valid_docs)}")

embedding = OpenAIEmbeddings(
    model="text-embedding-v3",
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
    check_embedding_ctx_length=False,
    chunk_size=10,
)

collection_name = "c_v7"
chroma_client = chromadb.HttpClient(host="127.0.0.1", port=18000)

# Check if collection exists to avoid redundant indexing
existing_collections = [c.name for c in chroma_client.list_collections()]

if collection_name in existing_collections:
    logger.info(
        f"Collection '{collection_name}' already exists. Loading existing index."
    )
    vectorstore = Chroma(
        client=chroma_client,
        collection_name=collection_name,
        embedding_function=embedding,
    )
else:
    logger.info(f"Collection '{collection_name}' not found. Creating new index.")
    vectorstore = Chroma.from_documents(
        documents=valid_docs,
        embedding=embedding,
        client=chroma_client,
        collection_name=collection_name,
    )

2026-01-26 17:19:38.469 | INFO     | __main__:<module>:5 - Total splits: 214, Valid splits: 214
2026-01-26 17:19:41.878 | INFO     | __main__:<module>:22 - Collection 'c_v7' already exists. Loading existing index.
C:\Users\hzr.HZR14635-PC\AppData\Local\Temp\ipykernel_40132\1475369280.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [10]:
retriever = vectorstore.as_retriever()
logger.success(f"Vector store connected to persistent server (Port 18000)")

2026-01-26 17:20:25.408 | SUCCESS  | __main__:<module>:2 - Vector store connected to persistent server (Port 18000)


In [11]:
llm = ChatOpenAI(
    model="deepseek-v3.2",
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE
)

In [12]:
query = "What is the Long-term memory?"
retrived_docs = retriever.invoke(query)

## Native

In [14]:
template = """
Anwser the question based on the context below.
Context:
{context}
Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = prompt | llm
repsonse = chain.invoke({"context": retrived_docs, "question": query})

logger.info(repsonse.content)

2026-01-26 17:21:18.083 | INFO     | __main__:<module>:14 - Long-term memory is a component that enables an agent to retain and recall vast amounts of information over extended periods. It typically functions as an external vector store, allowing fast retrieval when needed. This memory type has an essentially unlimited storage capacity and can last from a few days to decades.


## 1. Multi-Query

### 1.1 Core Principle: Solving the "Instability" of Vector Space

Why is one question not enough?

*   **Vector Sensitivity**: Embedding models are extremely sensitive to wording. For example, "how to solve error A" and "troubleshooting methods for A" mean the same thing, but their coordinates in vector space may have significant bias.
*   **Retrieval Limitations**: A single retrieval (Top-K) only finds chunks closest to that specific point. If the correct answer is slightly farther away, it will be missed.

**The Multi-Query Approach:** Leverage the linguistic power of LLMs to generate **3-5 variations** of the same query with different phrasing. This effectively drops multiple coordinates on the vector map simultaneously, searching their respective neighborhoods.

![Multi-Query](../../../assets/images/03_Engineering/04_RAG/multi_query.png)

### 1.2 Standard Workflow

1.  **Query Rewrite**: LLM takes the original question $Q$ and outputs $Q_1, Q_2, Q_3$ as diverse variants.
2.  **Parallel Retrieval**: The system performs simultaneous searches in the vector store for $Q, Q_1, Q_2, Q_3$.
3.  **Result Union**: All document chunks returned from the multiple queries are aggregated into a single collection.
4.  **De-duplication**: Since the same document might be retrieved by multiple queries, duplicates are filtered out.
5.  **Final Generation**: The de-duplicated relevant chunks (Context) are fed to the LLM to generate the final answer.

In [17]:
template = """You are an expert in LLM architecture, RAG systems, and AI memory mechanisms. Your task is to generate {k} 
different versions of the given user question to retrieve relevant documents from a vector database. 
By generating multiple perspectives on the user question, your goal is to help 
overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.
Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(
    model="deepseek-v3.2",
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
    temperature=0.4,
)

generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

query = "What is the Long-term memory?"
queries = generate_queries.invoke({"question": query, "k": 5})
for query in queries:
    logger.info(query)

2026-01-26 19:19:51.615 | INFO     | __main__:<module>:23 - 1. Explain the concept of long-term memory in the context of artificial intelligence and large language models.
2026-01-26 19:19:51.619 | INFO     | __main__:<module>:23 - 2. How do AI systems, particularly LLMs, implement and utilize long-term memory mechanisms?
2026-01-26 19:19:51.632 | INFO     | __main__:<module>:23 - 3. Describe architectural approaches for persistent memory in RAG systems and conversational agents.
2026-01-26 19:19:51.639 | INFO     | __main__:<module>:23 - 4. What are the methods for enabling an LLM to retain and recall information across multiple sessions or interactions?
2026-01-26 19:19:51.647 | INFO     | __main__:<module>:23 - 5. Compare and contrast short-term context windows with long-term memory solutions in modern AI architectures.


## 2. RAG Fusion

In [ ]:
def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)
    return [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

## 3. Decomposition

In [ ]:
template = """You are an AI language model assistant. Your task is to extract 
foundational sub-questions required to answer the following user question.
Provide these sub-questions separated by newlines.
Original question: {question}"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

generate_sub_questions = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

## 4. Step Back

In [ ]:
system = """You are an expert at world knowledge. Your task is to step back and 
generate a more general search query that could help answer a complex specific search query.
Input Query: {question}
Step Back Query: """
prompt_step_back = ChatPromptTemplate.from_template(system)
generate_step_back = prompt_step_back | llm | StrOutputParser()

## 5. HyDE

In [ ]:
# HyDE: Generates a hypothetical answer
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""

prompt_hyde = ChatPromptTemplate.from_template(template)
generate_hypothetical_doc = (
    prompt_hyde | llm | StrOutputParser()
)

question = "How do LLM-based agents manage long-term memory?"
hyde_chain = generate_hypothetical_doc | retriever
results = hyde_chain.invoke({"question": question})
logger.info(f"HyDE retrieved {len(results)} relevant documents using a hypothetical answer.")